# Securing Couchbase MCP Server with Keycloak — Non-DCR and DCR Flows

This tutorial showcases how to set-up the Couchbase MCP server in Streamable HTTP mode with OAuth settings so that it serves clients that operate in the Browser flow (DCR and Non-DCR) using Keycloak as the identity provider:

- **Non-DCR flow** — pre-registered client with browser login, tested with MCP Inspector and VS Code
- **DCR flow** — dynamic client registration with browser login, tested with MCP Inspector, VS Code, and Cursor

> ℹ️ **Non-DCR vs DCR?** In Non-DCR (non-Dynamic Client Registration), the client app is manually pre-registered in Keycloak before connecting. In DCR ([RFC 7591](https://datatracker.ietf.org/doc/html/rfc7591)), the client registers itself automatically at connection time — no manual setup needed. DCR is more flexible but requires additional Keycloak configuration to allow anonymous registration.

> 📖 You can read more about [Couchbase MCP Server OAuth Authentication](https://mcp-server.couchbase.com/configuration/oauth) and the [Keycloak documentation](https://www.keycloak.org/documentation).

## Prerequisites

- Docker installed (to run Keycloak locally)
- MCP Inspector (`npx @modelcontextprotocol/inspector`) or an IDE like VS Code.
- A running Couchbase cluster with credentials

## What to expect

This tutorial is organized into three steps:

**Step 1 — Keycloak and MCP server setup (shared foundation for both flows)**

This step sets up everything needed before any client can connect — the identity provider configuration and the MCP server itself.

- Set up a dedicated Keycloak realm to isolate MCP configuration from other applications
- Create custom OAuth scopes (`couchbase-mcp:read` and `couchbase-mcp:write`) that the MCP server uses for per-tool access enforcement
- Register the Couchbase MCP server as a resource server in Keycloak, defining the audience and scope contract for issued tokens
- Create a test user for browser-based login flows
- Start the MCP server with OAuth and PRM (Protected Resource Metadata) enabled so clients can discover the authorization server automatically

**Step 2 — Non-DCR flow (pre-registered client, required for governance-controlled access)**

This step is for scenarios where client access must be explicitly controlled — for example, granting read-only access to some clients and read-write to others. Each client is manually registered in Keycloak before connecting.

- Pre-register an OAuth client in Keycloak with the appropriate scopes and redirect URIs
- Validate the connection using MCP Inspector (providing the client ID in auth settings) and VS Code (prompted for client ID on connect)

> ℹ️ Non-DCR is the recommended approach when governance or access control matters — it lets administrators control exactly which scopes each client can request, by registering separate clients per access level (e.g. `mcp-client-read` vs `mcp-client-readwrite`).

**Step 3 — DCR flow (dynamic client registration, zero pre-configuration per client)**

This step is for scenarios where clients should be able to connect without any prior registration — Keycloak assigns a client ID automatically at connection time.

- Enable anonymous Dynamic Client Registration (DCR) in Keycloak
- Validate that clients can self-register and connect without a pre-registered client ID, using MCP Inspector and VS Code

> ℹ️ DCR is simpler to operate but offers less granular access control — all dynamically registered clients receive the same default scopes. Use Non-DCR if per-client scope restrictions are required.

---

## Prerequisite

For this tutorial, Keycloak is run locally to demonstrate the tutorial using docker, by running the following:

```bash
docker run -p 8080:8080 \
  -e KC_BOOTSTRAP_ADMIN_USERNAME=admin \
  -e KC_BOOTSTRAP_ADMIN_PASSWORD=admin \
  quay.io/keycloak/keycloak:latest start-dev
```

Open http://localhost:8080 and log in with `admin` / `admin`.

Keycloak might already be running in your case.



<img src="keycloak_screenshots/key_1.png" width="500">

---

## Step 1 — Keycloak Setup (shared by both flows)

### Step 1.1 — Create a Realm

A Realm in Keycloak is an isolated security domain — it has its own set of users, clients, scopes, and configurations. Think of it as a tenant. Creating a dedicated realm for MCP keeps your MCP configuration cleanly separated from any other applications in your Keycloak instance. See Keycloak Realm concepts for more.

- Click the top-left dropdown → **Create Realm**
- Name it `mcp-realm` → **Create**

<img src="keycloak_screenshots/key_2.png" width="500">

### Step 1.2 — Create custom scopes

Client Scopes in Keycloak define the permissions that can appear in an access token's scope claim. The Couchbase MCP server uses two scopes to enforce per-tool access control — `couchbase-mcp:read` for read-only tools and `couchbase-mcp:write` for mutation tools. See Keycloak Client Scopes documentation for more.

- Left nav → **Client Scopes** → **Create client scope**
- Create `couchbase-mcp:read`:
  - Name: `couchbase-mcp:read`
  - Type: `Optional`
  - Protocol: `openid-connect`
  - **Include in token scope** → toggle **On** — this makes the scope name appear in the scope claim of the token. Without this, the scope is assigned but never shows up in the token even if the client requests it.
  - **Save**
- Repeat for `couchbase-mcp:write`

<img src="keycloak_screenshots/key_3.png" width="500">

### Step 1.3 — Register Couchbase MCP Server as a Resource Server in Keycloak

> ℹ️ **Keycloak's "Client" terminology:** In Keycloak, everything is called a "Client" — both the application requesting tokens (the OAuth client) and the API being protected (the resource server). This step creates a Keycloak Client that represents the resource server — i.e. the Couchbase MCP server itself. Its purpose is to define the audience (`aud` claim) and the scopes that tokens must carry to access it. This is different from the OAuth client you'll create in Step 2.1, which represents the browser-based application that actually requests tokens. See Keycloak Client concepts for more.

- Left nav → **Clients** → **Create client**
- Client ID: `couchbase-mcp-server`
- Client authentication: **Off** (public) — resource servers don't authenticate to Keycloak, they only validate incoming tokens
- Click through to **Save**
- Go to **Client Scopes** tab → **Add client scope** → add both `couchbase-mcp:read` and `couchbase-mcp:write` as **Optional**

<img src="keycloak_screenshots/key_4.png" width="500">

<img src="keycloak_screenshots/key_5.png" width="500">

<img src="keycloak_screenshots/key_6.png" width="500">

### Step 1.4 — Add audience mapper

By default, Keycloak does not include the resource server's client ID in the token's `aud` (audience) claim. The MCP server validates the `aud` claim to ensure the token was issued specifically for it — without this mapper, every token will be rejected with an audience mismatch error. See Keycloak Audience Support for more.

- Go to **Client Scopes** → `couchbase-mcp:read` → **Mappers** tab → **Add mapper** → **By configuration** → **Audience**
- Fill in:
  - Name: `couchbase-mcp-audience`
  - Included Client Audience: `couchbase-mcp-server`
  - Add to access token: **On**
  - **Save**
- Repeat on `couchbase-mcp:write` scope

<img src="keycloak_screenshots/key_10.png" width="500">

### Step 1.5 — Create a test user

- Left nav → **Users** → **Add user**
- Fill in:
  - Username: `testuser`
  - First name: `Test`
  - Last name: `User`
  - Email: `testuser@test.com`
- **Details** tab → toggle **Email verified** → **On** → **Save** — without this the login flow returns `Account is not fully set up`
- **Credentials** tab → **Set password** → `password` → turn off **Temporary** → **Save**

> ⚠️ First name, last name, and email must be filled in — Keycloak blocks login if the user profile is incomplete.

<img src="keycloak_screenshots/key_11.png" width="500">

<img src="keycloak_screenshots/key_7.png" width="500">

### Step 1.6 — Collect server config values

These values are used in the MCP server startup command. The format shown below is for a local Keycloak instance running on port 8080. For other deployments, replace `http://localhost:8080` with your Keycloak base URL:

- **Standalone server:** use the hostname and port Keycloak is configured to serve on, e.g. `https://keycloak.yourdomain.com`
- **Kubernetes:** use the ingress or service URL, e.g. `https://keycloak.internal.example.com`
- **Managed service** (Red Hat SSO, etc.): use the tenant URL provided by your service

The realm name (`mcp-realm`) stays the same regardless of deployment.

| Value | Local format | How to find it for other deployments |
| --- | --- | --- |
| Issuer | `http://localhost:8080/realms/mcp-realm` | `<keycloak-base-url>/realms/<realm-name>` — also visible in the realm's OpenID Connect Discovery document at `<issuer>/.well-known/openid-configuration` |
| JWKS URI | `http://localhost:8080/realms/mcp-realm/protocol/openid-connect/certs` | `<issuer>/protocol/openid-connect/certs` |
| Audience | `couchbase-mcp-server` | Always the Client ID of the resource server client created in Step 1.3 — same regardless of deployment |

---

#### For Non-DCR flow, follow **Step 2 - Non-DCR Flow**
#### For DCR flow, follow **Step 2 - DCR Flow** (skip Step 2 - Non-DCR Flow)


## Step 2 — Non-DCR Flow (Pre-registered Client)

This flow uses a pre-registered public client with a browser-based authorization code + PKCE flow. The client ID is provided upfront, and Keycloak handles the browser login and token issuance.

### Step 2.1 — Create the pre-registered client

This creates the OAuth client for the Non-DCR flow — the browser-based application (MCP Inspector or VS Code) that drives the authorization code + PKCE flow. Unlike the resource server in Step 1.3, this client has Standard flow enabled and redirect URIs configured because it participates in the browser login redirect cycle. See Keycloak PKCE documentation for more.

- Left nav → **Clients** → **Create client**
- Client ID: `mcp-inspector-client`
- Client authentication: **Off** (public) — this is a browser-based public client; secrets cannot be safely stored in a browser or desktop app, so PKCE is used instead to secure the flow
- Authentication flow: check **Standard flow** only
- Valid redirect URIs — add all of the following:
  - `http://localhost:6274/oauth/callback` — MCP Inspector
  - `http://127.0.0.1:33418/` — VS Code 
  - `https://vscode.dev/redirect` — VS Code (web)
- Web origins (needed for MCP Inspector's browser-based CORS requests):
  - `http://localhost:6274`
  - `http://127.0.0.1:6274`
- **Save**
- Go to **Settings** tab → scroll to **Require PKCE** → toggle **On** → **Save** — this enforces PKCE for all authorization requests from this client, which is best practice for public clients per OAuth 2.0 Security Best Current Practice. MCP Inspector and VS Code both support PKCE natively.
- **Client Scopes** tab → add `couchbase-mcp:read` and `couchbase-mcp:write` as **Default**

<img src="keycloak_screenshots/key_15.png" width="500">

<img src="keycloak_screenshots/key_22.png" width="500">

<img src="keycloak_screenshots/key_37.png" width="500">

<img src="keycloak_screenshots/key_16.png" width="500">

### Step 2.2 — Start the MCP server with PRM

In [ ]:
uvx couchbase-mcp-server \
  --transport=http \
  --connection-string="couchbase://127.0.0.1" \
  --username="<your-couchbase-username>" \
  --password="<your-couchbase-password>" \
  --read-only-mode=false \
  --oauth-jwks-uri="http://localhost:8080/realms/mcp-realm/protocol/openid-connect/certs" \
  --oauth-issuer="http://localhost:8080/realms/mcp-realm" \
  --oauth-audience="couchbase-mcp-server" \
  --oauth-mcp-base-url="http://127.0.0.1:8000"

Verify the PRM document:

In [ ]:
curl -s http://127.0.0.1:8000/.well-known/oauth-protected-resource/mcp | python3 -m json.tool

Expected response:

```json
{
  "resource": "http://127.0.0.1:8000/mcp",
  "authorization_servers": ["http://localhost:8080/realms/mcp-realm"],
  "scopes_supported": ["couchbase-mcp:read", "couchbase-mcp:write"]
}
```


### Step 2.3 — Configure MCP Inspector

```bash
npx @modelcontextprotocol/inspector
```

In the Inspector UI:

| Field | Value |
| --- | --- |
| Transport Type | `Streamable HTTP` |
| URL | `http://127.0.0.1:8000/mcp` |
| Client ID | `mcp-inspector-client` |
| Client Secret | (leave blank) |
| Redirect URL | `http://localhost:6274/oauth/callback` |
| Scope | (leave blank) |

Authorization URL and Token URL are auto-discovered from the PRM document.


### Step 2.4 — Connect and verify via MCP Inspector

- Click **Connect**
- A browser window opens with the Keycloak login page
- Log in with `testuser` / `password`
- Inspector shows "Successfully authenticated with OAuth"

Verify:

- **Tools** tab → **List Tools** — all 24 tools visible
- Run a read tool → success
- Run a write tool → success

<img src="keycloak_screenshots/key_17.png" width="500">

<img src="keycloak_screenshots/key_18.png" width="500">

### Step 2.5 — Connect via an IDE

#### VS Code

In VS Code, add the MCP server to `mcp.json` with no token — VS Code will prompt for the client ID and drive the auth code + PKCE flow automatically:

```json
{
  "servers": {
    "couchbase-keycloak-nondcr": {
      "type": "http",
      "url": "http://127.0.0.1:8000/mcp"
    }
  }
}
```

When VS Code prompts for a Client ID, enter `mcp-inspector-client`. A browser window will open with the Keycloak login page → log in with `testuser` / `password` → VS Code connects.

<img src="keycloak_screenshots/key_19.png" width="500">

<img src="keycloak_screenshots/key_20.png" width="500">

<img src="keycloak_screenshots/key_21.png" width="500">

<img src="keycloak_screenshots/key_24.png" width="500">

---

## Step 2 — DCR Flow (Dynamic Client Registration)

This flow requires no pre-registered client ID. The MCP client discovers Keycloak via the PRM document and self-registers anonymously at connection time.

> ℹ️ If you skipped Step 2 - Non-DCR Flow, start here. All you need from Step 1 is already set up — no client pre-registration is required for DCR.

### Step 2.1 — Start the MCP server with PRM

> ℹ️ If you completed Step 2, the MCP server is already running — skip to Step 3.2.

In [ ]:
uvx couchbase-mcp-server \
  --transport=http \
  --connection-string="couchbase://127.0.0.1" \
  --username="<your-couchbase-username>" \
  --password="<your-couchbase-password>" \
  --read-only-mode=false \
  --oauth-jwks-uri="http://localhost:8080/realms/mcp-realm/protocol/openid-connect/certs" \
  --oauth-issuer="http://localhost:8080/realms/mcp-realm" \
  --oauth-audience="couchbase-mcp-server" \
  --oauth-mcp-base-url="http://127.0.0.1:8000"

> ℹ️ Replace `http://localhost:8080` with your Keycloak base URL if not running locally.

Verify the PRM document

In [ ]:
curl -s http://127.0.0.1:8000/.well-known/oauth-protected-resource/mcp | python3 -m json.tool

Expected response:

```json
{
  "resource": "http://127.0.0.1:8000/mcp",
  "authorization_servers": ["http://localhost:8080/realms/mcp-realm"],
  "scopes_supported": ["couchbase-mcp:read", "couchbase-mcp:write"]
}
```


### Step 2.2 — Enable anonymous DCR in Keycloak

Dynamic Client Registration (DCR) allows MCP clients to register themselves with Keycloak at connection time without any manual pre-configuration. The client sends a registration request to Keycloak's `/clients-registrations/openid-connect` endpoint, receives a `client_id` back, and immediately uses it to start the authorization code flow. This is defined in [RFC 7591](https://datatracker.ietf.org/doc/html/rfc7591) and Keycloak's DCR documentation. By default Keycloak locks down DCR — you need to configure it to allow anonymous registration.

- Make sure you're in `mcp-realm`
- Left nav → **Clients** → **Client registration** tab → **Client registration policies** sub-tab
- Under **Anonymous Access Policies**, find **Client Disabled Policy** → **Delete** it — it auto-disables newly registered clients which breaks the flow

- Under **Anonymous Access Policies**, find **Trusted Hosts** → **Delete** it entirely. This is the recommended approach for local dev/testing — different MCP clients (Inspector, VS Code, Cursor) send DCR requests from different hosts and IPs, making it impractical to maintain a fixed allowlist.

> ℹ️ If you want to restrict which hosts can register clients (e.g. in a production or shared environment), keep the Trusted Hosts policy instead of deleting it and add the specific hosts: `localhost`, `127.0.0.1`, `192.168.65.1` (Docker bridge network IP — Keycloak in Docker sees requests from your Mac as this IP), and any remote hosts. Check **Events** in Keycloak for `CLIENT_REGISTER_ERROR` entries to see exactly which IP is being rejected if a client fails to register.

<img src="keycloak_screenshots/key_36.png" width="500">

- Trusted redirect URIs — leave empty to allow any redirect URI, recommended for dev/test since Cursor uses a dynamic port. If you want to restrict to specific clients:
  - `http://localhost:6274/oauth/callback` — MCP Inspector
  - `http://127.0.0.1:33418` — VS Code
  - `https://vscode.dev/redirect` — VS Code (web)

### Step 2.3 — Allow anonymous clients to request custom scopes

By default Keycloak blocks anonymous DCR clients from requesting arbitrary scopes.

- Still in **Client registration policies** → **Allowed Client Scopes** (under Anonymous Access Policies)
- Add all three scopes to the allowed scopes list:
  - `openid`
  - `couchbase-mcp:read`
  - `couchbase-mcp:write`
- **Save**

> ⚠️ `openid` must be included — even though it's a standard scope, Keycloak's Allowed Client Scopes policy blocks it if not explicitly listed. Without it, DCR requests that include `openid` in the scope will be rejected with `Policy 'Allowed Client Scopes' rejected request`.

### Step 2.4 — Verify DCR is working

Before connecting with Inspector, confirm the DCR endpoint accepts anonymous registration:

In [ ]:
curl -X POST \
  http://127.0.0.1:8080/realms/mcp-realm/clients-registrations/openid-connect \
  -H "Content-Type: application/json" \
  -d '{
    "client_name": "test-dcr-client",
    "redirect_uris": ["http://127.0.0.1:6274/oauth/callback"],
    "grant_types": ["authorization_code"],
    "response_types": ["code"],
    "token_endpoint_auth_method": "none"
  }'

If you get back a JSON with a `client_id` → DCR is working.

> ℹ️ No `scope` in the DCR request — Keycloak automatically assigns the default scopes (including `couchbase-mcp:read` and `couchbase-mcp:write`) to DCR clients because Allow Default Scopes is On. If you explicitly pass `scope` in the DCR payload, the Allowed Client Scopes policy will reject it. Leave `scope` out and let Keycloak assign it automatically.

### Step 2.5 — Start the MCP server

Same server command as Non-DCR:

In [ ]:
uvx couchbase-mcp-server \
  --transport=http \
  --connection-string="couchbase://127.0.0.1" \
  --username="<your-couchbase-username>" \
  --password="<your-couchbase-password>" \
  --read-only-mode=false \
  --oauth-jwks-uri="http://localhost:8080/realms/mcp-realm/protocol/openid-connect/certs" \
  --oauth-issuer="http://localhost:8080/realms/mcp-realm" \
  --oauth-audience="couchbase-mcp-server" \
  --oauth-mcp-base-url="http://127.0.0.1:8000"

### Step 2.6 — Configure MCP Inspector to validate DCR flow

```bash
npx @modelcontextprotocol/inspector
```

In the Inspector UI:

| Field | Value |
| --- | --- |
| Transport Type | `Streamable HTTP` |
| URL | `http://127.0.0.1:8000/mcp` |
| Client ID | (leave blank) |
| Client Secret | (leave blank) |
| Redirect URL | `http://localhost:6274/oauth/callback` |
| Scope | (leave blank) |

Inspector will hit the PRM, discover Keycloak, call the DCR endpoint anonymously, get a `client_id`, then drive the auth code + PKCE flow automatically.


### Step 2.7 — Connect and verify via MCP Inspector

- Click **Connect**
- Inspector calls `http://localhost:8080/realms/mcp-realm/clients-registrations/openid-connect` → gets a new `client_id`
- A browser window opens with the Keycloak login page
- Log in with `testuser` / `password`
- Inspector shows "Successfully authenticated with OAuth"

<img src="keycloak_screenshots/key_29.png" width="500">

<img src="keycloak_screenshots/key_30.png" width="500">

**Verify**:

- **Tools** tab → **List Tools** — all 24 tools visible
- Run a read tool → success
- Run a write tool → success

<img src="keycloak_screenshots/key_32.png" width="500">

### Step 2.8 — Connect via an IDE

#### VS Code

In VS Code, add the MCP server to `mcp.json` with no token and no client ID — VS Code drives the full DCR flow automatically:

```json
{
  "servers": {
    "couchbase-keycloak-dcr": {
      "type": "http",
      "url": "http://127.0.0.1:8000/mcp"
    }
  }
}
```

On connect, VS Code will:

- Hit the MCP server → get a `401`
- Read the PRM document → discover Keycloak
- Call the DCR endpoint → self-register → get a new `client_id`
- Open a browser to Keycloak's `/authorize`
- Log in with `testuser` / `password`
- VS Code receives the JWT → connected

#### Cursor

In Cursor, to open the MCP configuration file, go to settings and search for `Tools & MCPs`. Click on `New MCP Server`, this opens mcp.json. Add the following:

```json
{
  "mcpServers": {
    "couchbase-keycloak-dcr": {
      "type": "http",
      "url": "http://127.0.0.1:8000/mcp"
    }
  }
}
```

Go back to `Tools & MCPs` in the settings and click on `Connect` next to the couchbase MCP. Grant Access to Keycloak and complete the Authentication.

<img src="keycloak_screenshots/key_33.png" width="500">

<img src="keycloak_screenshots/key_34.png" width="500">

Ask the chat to use one of the MCP tools to validate.


<img src="keycloak_screenshots/key_35.png" width="500">
